<a href="https://colab.research.google.com/github/rmndrs89/advanced-time-series-prediction/blob/main/3_Model/DeepConvLSTM_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup the notebook

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# @title Import libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.model_selection import train_test_split

In [10]:
class FOGDataset(Dataset):
    def __init__(self, metadata, data_dir, mean=None, std=None):
        self.metadata = metadata.reset_index(drop=True)
        self.data_dir = data_dir
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        file = self.metadata.iloc[idx]["file_name"]
        label = int(self.metadata.iloc[idx]["label"])
        path = f"{self.data_dir}/{file}"
        with open(path, 'rb') as infile:
            data = np.load(infile)
        data = data[:, 1:4].T  # (3, 1280)

        if self.mean is not None and self.std is not None:
            data = (data - self.mean[:, None]) / self.std[:, None]

        return torch.tensor(data, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

In [16]:
class DeepConvLSTM(nn.Module):
    def __init__(self, input_channels=3, conv_filters=64, lstm_hidden=128, num_classes=2):
        super(DeepConvLSTM, self).__init__()

        self.batchnorm1 = nn.BatchNorm1d(input_channels)
        self.conv1 = nn.Conv1d(input_channels, conv_filters, kernel_size=5, padding=2)
        self.batchnorm2 = nn.BatchNorm1d(conv_filters)

        self.lstm = nn.LSTM(input_size=conv_filters, hidden_size=lstm_hidden, num_layers=2, batch_first=True)

        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.batchnorm1(x)
        x = self.conv1(x)
        x = self.batchnorm2(x)
        x = x.permute(0, 2, 1)
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]
        return self.classifier(out)


In [11]:
DATA_PATH = Path("/content/drive/MyDrive/Datasets/tdcsfog")
TRAIN_PATH = DATA_PATH / "train"
TEST_PATH = DATA_PATH / "test"
MODEL_PATH = DATA_PATH / "models"
OUTPUT_PATH = DATA_PATH / "outputs"

metadata = pd.read_csv(DATA_PATH / "test.csv")

In [18]:
model = DeepConvLSTM()
model.load_state_dict(torch.load(OUTPUT_PATH / "best_model.pt", map_location=torch.device("cpu"), weights_only=True))
model.eval()

DeepConvLSTM(
  (batchnorm1): BatchNorm1d(3, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv1): Conv1d(3, 64, kernel_size=(5,), stride=(1,), padding=(2,))
  (batchnorm2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True)
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=2, bias=True)
  )
)

In [13]:
with open(DATA_PATH / "train_config.npy", 'rb') as infile:
    config = np.load(infile)
    mean = config["mean"]
    std = config["std"]

test_dataset = FOGDataset(
    metadata=metadata,
    data_dir=TEST_PATH,
    mean=mean,
    std=std,
)

test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [20]:
total_loss, correct, total = 0, 0, 0

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        output = model(batch_x)
        _, predicted = torch.max(output.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

In [21]:
correct / total

0.7427083333333333